In [ ]:
import duckdb, requests
import pandas as pd

In [20]:
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def urls(table: str, limit: int = 50) -> list[str]:
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
                 if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    return file_list[-limit:]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")

file_urls = urls("match_player", limit=30)
con.execute(f"CREATE VIEW match_player AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

In [21]:
match_player_cols = con.sql("DESCRIBE match_player;").df()["column_name"]
print(match_player_cols)
print(match_player_cols.shape)

0                               match_id
1                             start_time
2                             duration_s
3                             match_mode
4                              game_mode
                     ...                
152    power_up_buffs.pickup_stat_values
153          hero_release_votes.category
154           hero_release_votes.hero_id
155        hero_release_votes.vote_count
156             custom_user_stats_deltas
Name: column_name, Length: 157, dtype: object
(157,)


In [60]:
match_query = """
SELECT DISTINCT ON (match_id) account_id, match_id, start_time
   FROM match_player
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(match_query)
print("Finding random match info including account_id")
display(con.sql(match_query).df())

Finding random match info including account_id


,account_id,match_id,start_time
0,104124469,123999990,2026-10-24 17:56:05-04:00
1,30057498,123999994,2026-10-24 17:56:05-04:00
2,147993254,123999988,2026-10-24 17:56:05-04:00
3,106594711,123999986,2026-10-24 17:56:05-04:00
4,5856312,123999979,2026-10-24 17:56:04-04:00


In [59]:
kd_query = """
SELECT account_id, "stats.kills", kills, "stats.deaths", deaths, start_time
   FROM match_player
   WHERE account_id = 147993254
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(kd_query)
print("Testing KD data with random account info")
display(con.sql(kd_query).df())

Testing KD data with random account info


,account_id,stats.kills,kills,stats.deaths,deaths,start_time
0,147993254,"[0, 0, 0, 1, 3, 4]",4,"[0, 1, 1, 1, 1, 2]",2,2026-10-24 17:56:05-04:00
1,147993254,"[0, 0, 0, 0, 0, 1]",1,"[0, 0, 0, 0, 2, 2]",2,2026-10-24 17:35:50-04:00
2,147993254,"[0, 1, 1, 1, 1, 1]",1,"[1, 1, 1, 1, 2, 4]",4,2026-10-23 21:11:18-04:00


In [58]:
stat_query = """
SELECT account_id, "stats.kills", "stats.deaths", "objectives.team_objective", start_time
   FROM match_player
   WHERE account_id = 147993254
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(stat_query)
print("Testing objectives compared to kills and death data with random account info")
display(con.sql(stat_query).df())

Testing objectives compared to kills and death data with random account info


,account_id,stats.kills,stats.deaths,objectives.team_objective,start_time
0,147993254,"[0, 0, 0, 1, 3, 4]","[0, 1, 1, 1, 1, 2]","[Tier2Lane1, Tier2Lane4, Tier2Lane4, Tier2Lane...",2026-10-24 17:56:05-04:00
1,147993254,"[0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 2, 2]","[Tier2Lane1, Tier2Lane4, Tier2Lane4, Tier2Lane...",2026-10-24 17:35:50-04:00
2,147993254,"[0, 1, 1, 1, 1, 1]","[1, 1, 1, 1, 2, 4]","[Tier2Lane1, Tier2Lane4, Tier2Lane4, Tier2Lane...",2026-10-23 21:11:18-04:00


In [61]:
item_count = """
SELECT account_id,
len("stats.kills") as kills_count,
len("stats.deaths") as deaths_count,
len("objectives.team_objective") as team_objective_count,
   start_time
   FROM match_player
   WHERE account_id = 147993254
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(item_count)
print("Testing counts of items in lists with random account info")
display(con.sql(item_count).df())

Testing counts of items in lists with random account info


,account_id,kills_count,deaths_count,team_objective_count,start_time
0,147993254,6,6,12,2026-10-24 17:56:05-04:00
1,147993254,6,6,12,2026-10-24 17:35:50-04:00
2,147993254,6,6,12,2026-10-23 21:11:18-04:00


Kills and deaths seem to match eachother on timelines. Thought they may line up with objective but this is not the case. Because these have an associated time of kill, and time of death it should be possible to use temporally for modelling. I wanted to check this just to see if the data was randomly pulled at different intervals. This does seem to be the case, as deaths jump from 1 to 3 as seen before, but these timeframes most likely don't associate with the objective due to the count not matching.

In [63]:
match_test = """
SELECT account_id,
len("stats.kills") as kills_count, "stats.kills",
len("stats.deaths") as deaths_count, "stats.deaths",
"stats.time_stamp_s", len("stats.time_stamp_s") as time_stamp_count,
match_id, duration_s
   FROM match_player
   WHERE match_id = ?
   GROUP BY account_id, "stats.kills", "stats.deaths", "stats.time_stamp_s", match_id, duration_s
   """

match_list = [123999988, 123999990, 123999986, 123999994, 123999979]

for match in match_list:
   print(f"\nTesting counts of items in lists with random match info for match_id: {match}")
   
   con.execute(match_test, [match])
   result_df = con.df()
   display(result_df)


Testing counts of items in lists with random match info for match_id: 123999988


,account_id,kills_count,stats.kills,deaths_count,stats.deaths,stats.time_stamp_s,time_stamp_count,match_id,duration_s
0,147993254,6,"[0, 0, 0, 1, 3, 4]",6,"[0, 1, 1, 1, 1, 2]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
1,184706545,6,"[0, 0, 0, 0, 0, 0]",6,"[1, 3, 5, 5, 7, 8]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
2,1158997793,6,"[0, 3, 4, 4, 4, 4]",6,"[0, 0, 0, 0, 1, 1]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
3,1298005728,6,"[2, 2, 4, 5, 5, 5]",6,"[0, 0, 0, 1, 2, 2]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
4,452598417,6,"[0, 0, 0, 0, 0, 0]",6,"[1, 2, 3, 5, 6, 7]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
5,855131750,6,"[0, 0, 0, 1, 1, 3]",6,"[0, 0, 0, 1, 1, 2]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
6,1322630583,6,"[0, 0, 0, 1, 1, 2]",6,"[1, 2, 3, 3, 3, 4]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185
7,1743548035,6,"[1, 3, 4, 4, 7, 9]",6,"[0, 0, 0, 0, 0, 1]","[180, 360, 540, 720, 900, 1185]",6,123999988,1185



Testing counts of items in lists with random match info for match_id: 123999990


,account_id,kills_count,stats.kills,deaths_count,stats.deaths,stats.time_stamp_s,time_stamp_count,match_id,duration_s
0,193233083,6,"[1, 4, 6, 8, 9, 12]",6,"[1, 1, 1, 2, 3, 5]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
1,197383378,6,"[1, 3, 3, 3, 4, 9]",6,"[1, 3, 5, 6, 6, 7]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
2,298555527,6,"[0, 1, 1, 1, 1, 1]",6,"[0, 2, 2, 4, 5, 6]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
3,357611973,6,"[0, 0, 0, 0, 0, 0]",6,"[1, 2, 2, 2, 3, 5]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
4,863774522,6,"[0, 0, 0, 1, 1, 2]",6,"[1, 2, 2, 2, 4, 6]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
5,104124469,6,"[0, 2, 4, 5, 5, 5]",6,"[1, 2, 2, 2, 3, 4]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
6,244918695,6,"[2, 2, 2, 2, 2, 2]",6,"[0, 1, 2, 3, 3, 4]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054
7,1107750259,6,"[1, 1, 1, 2, 6, 8]",6,"[0, 0, 1, 1, 1, 2]","[180, 360, 540, 720, 900, 1054]",6,123999990,1054



Testing counts of items in lists with random match info for match_id: 123999986


,account_id,kills_count,stats.kills,deaths_count,stats.deaths,stats.time_stamp_s,time_stamp_count,match_id,duration_s
0,151219612,10,"[0, 0, 0, 0, 0, 0, 1, 3, 3, 3]",10,"[0, 0, 0, 0, 1, 2, 4, 5, 5, 5]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
1,178628734,10,"[0, 0, 0, 0, 0, 4, 4, 4, 4, 5]",10,"[0, 2, 3, 3, 4, 4, 5, 7, 8, 9]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
2,398056392,10,"[1, 1, 3, 3, 3, 4, 5, 6, 6, 10]",10,"[1, 1, 2, 3, 4, 4, 7, 10, 11, 11]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
3,1152835222,10,"[0, 2, 2, 3, 5, 6, 7, 7, 10, 11]",10,"[0, 1, 2, 3, 3, 3, 4, 5, 5, 5]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
4,1902812404,10,"[0, 1, 3, 4, 5, 6, 6, 8, 9, 10]",10,"[0, 0, 1, 1, 1, 3, 3, 3, 3, 4]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
5,106594711,10,"[0, 0, 0, 0, 1, 1, 2, 2, 2, 2]",10,"[0, 1, 1, 1, 1, 2, 3, 4, 6, 7]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
6,236408542,10,"[0, 1, 1, 1, 1, 1, 1, 2, 2, 2]",10,"[1, 2, 2, 3, 3, 5, 6, 7, 7, 9]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
7,307863020,10,"[1, 1, 1, 1, 1, 3, 5, 6, 8, 12]",10,"[0, 0, 0, 1, 2, 2, 3, 5, 6, 6]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
8,340857428,10,"[1, 1, 1, 2, 2, 4, 4, 5, 6, 6]",10,"[0, 1, 2, 4, 6, 7, 9, 11, 11, 12]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233
9,1023487203,10,"[1, 1, 1, 4, 6, 6, 10, 11, 12, 12]",10,"[1, 1, 2, 3, 4, 5, 6, 8, 9, 11]","[180, 360, 540, 720, 900, 1200, 1500, 1800, 21...",10,123999986,2233



Testing counts of items in lists with random match info for match_id: 123999994


,account_id,kills_count,stats.kills,deaths_count,stats.deaths,stats.time_stamp_s,time_stamp_count,match_id,duration_s
0,30057498,7,"[0, 0, 0, 0, 0, 0, 0]",7,"[0, 0, 0, 0, 0, 0, 0]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
1,124304385,7,"[0, 0, 0, 0, 0, 0, 0]",7,"[1, 1, 1, 1, 2, 3, 3]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
2,296559157,7,"[0, 0, 0, 1, 1, 1, 1]",7,"[0, 0, 1, 1, 1, 1, 1]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
3,416716429,7,"[0, 0, 0, 0, 0, 0, 0]",7,"[0, 1, 1, 1, 2, 3, 4]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
4,1010555942,7,"[0, 0, 1, 1, 2, 4, 5]",7,"[0, 0, 1, 1, 1, 1, 1]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
5,1250957802,7,"[1, 2, 4, 5, 8, 10, 11]",7,"[0, 0, 0, 0, 0, 0, 0]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
6,1274983970,7,"[1, 2, 2, 2, 2, 2, 2]",7,"[0, 0, 3, 3, 4, 5, 6]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
7,1476194381,7,"[0, 2, 3, 4, 4, 6, 7]",7,"[0, 0, 1, 1, 1, 1, 2]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
8,1564787349,7,"[1, 1, 1, 1, 3, 7, 7]",7,"[1, 2, 2, 2, 2, 2, 2]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430
9,166325166,7,"[0, 0, 1, 1, 1, 1, 2]",7,"[1, 1, 2, 3, 5, 7, 8]","[180, 360, 540, 720, 900, 1200, 1430]",7,123999994,1430



Testing counts of items in lists with random match info for match_id: 123999979


,account_id,kills_count,stats.kills,deaths_count,stats.deaths,stats.time_stamp_s,time_stamp_count,match_id,duration_s
0,30776926,8,"[0, 0, 0, 0, 1, 6, 9, 12]",8,"[0, 0, 0, 0, 2, 2, 2, 2]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
1,172758328,8,"[0, 0, 0, 1, 1, 1, 1, 1]",8,"[0, 0, 1, 2, 4, 5, 7, 9]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
2,257865632,8,"[0, 0, 0, 0, 0, 0, 2, 2]",8,"[0, 0, 0, 0, 0, 1, 2, 2]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
3,344521177,8,"[0, 0, 0, 0, 0, 0, 1, 1]",8,"[0, 1, 2, 2, 4, 5, 8, 9]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
4,874751873,8,"[0, 1, 2, 2, 2, 2, 4, 5]",8,"[0, 0, 0, 1, 2, 2, 2, 3]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
5,1052946980,8,"[0, 0, 0, 1, 1, 2, 2, 2]",8,"[0, 0, 0, 1, 1, 1, 4, 4]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
6,5856312,8,"[1, 2, 3, 4, 5, 5, 5, 5]",8,"[0, 0, 0, 0, 1, 1, 3, 4]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
7,92962482,8,"[0, 1, 1, 1, 3, 4, 5, 6]",8,"[1, 1, 1, 3, 3, 5, 9, 10]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
8,181257123,8,"[0, 0, 2, 5, 7, 8, 10, 11]",8,"[0, 1, 1, 2, 3, 4, 5, 7]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703
9,244777931,8,"[0, 0, 1, 1, 3, 3, 5, 6]",8,"[1, 1, 2, 2, 2, 2, 3, 4]","[180, 360, 540, 720, 900, 1200, 1500, 1703]",8,123999979,1703


Tested to see if deaths and kills were pulled the same amount of times per game, this does not seem to be the case, but it does appear it may be related to the duration of the game

In [48]:
duration_Test = """
SELECT account_id,
len("stats.kills") as kills_count,
len("stats.deaths") as deaths_count,
match_id, duration_s,
duration_s / len("stats.kills") as durationPerReportedKill,
((duration_s / LEN("stats.kills")) / duration_s) * 100 AS percentOfGamePerReportedKill
   FROM match_player
   WHERE match_id = ?
   GROUP BY account_id, "stats.kills", "stats.deaths", match_id, duration_s
   """

match_list = [123999988, 123999990, 123999986, 123999994, 123999979]

for match in match_list:
   print(f"\nTesting the duration of time between reports of kills/deaths for match_id: {match}")
   
   con.execute(duration_Test, [match])
   result_df = con.df()
   display(result_df)


Testing the duration of time between reports of kills/deaths for match_id: 123999988


,account_id,kills_count,deaths_count,match_id,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,147993254,6,6,123999988,1185,197.5,16.666667
1,184706545,6,6,123999988,1185,197.5,16.666667
2,452598417,6,6,123999988,1185,197.5,16.666667
3,855131750,6,6,123999988,1185,197.5,16.666667
4,1158997793,6,6,123999988,1185,197.5,16.666667
5,1298005728,6,6,123999988,1185,197.5,16.666667
6,1322630583,6,6,123999988,1185,197.5,16.666667
7,1743548035,6,6,123999988,1185,197.5,16.666667



Testing the duration of time between reports of kills/deaths for match_id: 123999990


,account_id,kills_count,deaths_count,match_id,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,197383378,6,6,123999990,1054,175.666667,16.666667
1,244918695,6,6,123999990,1054,175.666667,16.666667
2,863774522,6,6,123999990,1054,175.666667,16.666667
3,104124469,6,6,123999990,1054,175.666667,16.666667
4,193233083,6,6,123999990,1054,175.666667,16.666667
5,298555527,6,6,123999990,1054,175.666667,16.666667
6,357611973,6,6,123999990,1054,175.666667,16.666667
7,1107750259,6,6,123999990,1054,175.666667,16.666667



Testing the duration of time between reports of kills/deaths for match_id: 123999986


,account_id,kills_count,deaths_count,match_id,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,106594711,10,10,123999986,2233,223.3,10.0
1,340857428,10,10,123999986,2233,223.3,10.0
2,1023487203,10,10,123999986,2233,223.3,10.0
3,1076464687,10,10,123999986,2233,223.3,10.0
4,1126345181,10,10,123999986,2233,223.3,10.0
5,1152835222,10,10,123999986,2233,223.3,10.0
6,1902812404,10,10,123999986,2233,223.3,10.0
7,151219612,10,10,123999986,2233,223.3,10.0
8,178628734,10,10,123999986,2233,223.3,10.0
9,236408542,10,10,123999986,2233,223.3,10.0



Testing the duration of time between reports of kills/deaths for match_id: 123999994


,account_id,kills_count,deaths_count,match_id,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,30057498,7,7,123999994,1430,204.285714,14.285714
1,124304385,7,7,123999994,1430,204.285714,14.285714
2,852303771,7,7,123999994,1430,204.285714,14.285714
3,1250957802,7,7,123999994,1430,204.285714,14.285714
4,1564787349,7,7,123999994,1430,204.285714,14.285714
5,166325166,7,7,123999994,1430,204.285714,14.285714
6,296559157,7,7,123999994,1430,204.285714,14.285714
7,416716429,7,7,123999994,1430,204.285714,14.285714
8,906104207,7,7,123999994,1430,204.285714,14.285714
9,1010555942,7,7,123999994,1430,204.285714,14.285714



Testing the duration of time between reports of kills/deaths for match_id: 123999979


,account_id,kills_count,deaths_count,match_id,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,5856312,8,8,123999979,1703,212.875,12.5
1,30776926,8,8,123999979,1703,212.875,12.5
2,92962482,8,8,123999979,1703,212.875,12.5
3,172758328,8,8,123999979,1703,212.875,12.5
4,181257123,8,8,123999979,1703,212.875,12.5
5,344521177,8,8,123999979,1703,212.875,12.5
6,1548743124,8,8,123999979,1703,212.875,12.5
7,244777931,8,8,123999979,1703,212.875,12.5
8,257865632,8,8,123999979,1703,212.875,12.5
9,874751873,8,8,123999979,1703,212.875,12.5


In [51]:
percentTest = """
SELECT DISTINCT match_id,
len("stats.kills") as kills_count,
len("stats.deaths") as deaths_count,
duration_s,
duration_s / len("stats.kills") as durationPerReportedKill,
   ((duration_s / LEN("stats.kills")) / duration_s) * 100 AS percentOfGamePerReportedKill
    FROM match_player
    GROUP BY match_id, "stats.kills", "stats.deaths", duration_s
    LIMIT 20"""

con.execute(percentTest)
print("Testing assumed length of time for the duration of time between reports of kills/deaths for match_id")
display(con.sql(percentTest).df())

Testing assumed length of time for the duration of time between reports of kills/deaths for match_id


,match_id,kills_count,deaths_count,duration_s,durationPerReportedKill,percentOfGamePerReportedKill
0,123132187,12,12,2742,228.500000,8.333333
1,123132188,12,12,2851,237.583333,8.333333
2,123132292,10,10,2230,223.000000,10.000000
3,123132419,5,5,721,144.200000,20.000000
4,123132495,4,4,551,137.750000,25.000000
5,123132617,6,6,923,153.833333,16.666667
6,123132704,9,9,1941,215.666667,11.111111
7,123132841,8,8,1634,204.250000,12.500000
8,123132991,10,10,2186,218.600000,10.000000
9,123133083,6,6,1106,184.333333,16.666667


In [57]:
averageTime = """
WITH match_metrics AS(
   SELECT DISTINCT match_id,
   len("stats.kills") as kills_count,
   len("stats.deaths") as deaths_count,
   duration_s,
   duration_s / len("stats.kills") as durationPerReportedKill,
   duration_s / len("stats.deaths") as durationPerReportedDeath,
   ((duration_s / LEN("stats.kills")) / duration_s) * 100 AS percentOfGamePerReportedKill,
   ((duration_s / LEN("stats.deaths")) / duration_s) * 100 AS percentOfGamePerReportedDeath
   
FROM match_player
GROUP BY match_id, "stats.kills", "stats.deaths", duration_s
LIMIT 250000
)
SELECT avg(durationPerReportedKill) as avg_durationPerReportedKill,
avg(durationPerReportedDeath) as avg_durationPerReportedDeath,
avg(percentOfGamePerReportedKill) as avg_percentOfGamePerReportedKill,
avg(percentOfGamePerReportedDeath) as avg_percentOfGamePerReportedDeath
FROM match_metrics
   """


con.execute(averageTime)
print("Finding average duration of time between reports of kills/deaths for match_id")
display(con.sql(averageTime).df())

Finding average duration of time between reports of kills/deaths for match_id


,avg_durationPerReportedKill,avg_durationPerReportedDeath,avg_percentOfGamePerReportedKill,avg_percentOfGamePerReportedDeath
0,214.120813,214.120813,11.957363,11.957363


From a subset of 250,000 matches, it appears that duration between "stats.kills" and "stats.deaths" being updated is 214.12 seconds on average, which is around 12% of the gametime. If we are modeling on values alone in these stats columns this is a good baseline to go off of.

In [ ]:
percentTest = """
SELECT DISTINCT match_id,
duration_s, "stats.time_stamp_s"
    FROM match_player
    WHERE duration_s > 2400
    GROUP BY match_id, duration_s, "stats.time_stamp_s"
    LIMIT 20"""

con.execute(percentTest)
print("Checking matches with duration greater than 2400 seconds to see if the time_stamp_s array is updated at a different rate")

pd.set_option('display.max_colwidth', None)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

display(con.sql(percentTest).df())

Testing assumed length of time for the duration of time between reports of kills/deaths for match_id


,match_id,duration_s,stats.time_stamp_s
0,123100735,2898,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2700, 2898]"
1,123100757,2764,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2700, 2764]"
2,123100768,2663,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2663]"
3,123100771,2511,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2511]"
4,123100797,2473,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2473]"
5,123100834,2484,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2484]"
6,123100837,2743,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2700, 2743]"
7,123100886,2565,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2565]"
8,123100893,2446,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2446]"
9,123100894,2536,"[180, 360, 540, 720, 900, 1200, 1500, 1800, 2100, 2400, 2536]"


"stats.time_stamp_s" contains an array of the timestamp for every "stats." array. From this information we can gather that these arrays are updated every 180 seconds until 900 seconds, when afterwards they proceed to only be updated about every 300 seconds. It is not clear if they are updated at different rates afterwards. Many games do not play past 40 minutes (2400 seconds) so this is not an issue but something to be aware of.